In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import distinctipy
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/age-regression"
df = pd.read_excel(f"{path_data}/data.xlsx")

df_groups = pd.read_excel(f"{path_data}/groups.xlsx", index_col=0)
icd_chpts = np.sort(df_groups['ICD-11 chapter'].unique())

# Prepare colors
def make_rgb_transparent(rgb, bg_rgb, alpha):
    return [alpha * c1 + (1 - alpha) * c2 for (c1, c2) in zip(rgb, bg_rgb)]

# Colors for ICD-11 chapters
colors = distinctipy.get_colors(len(icd_chpts), [mcolors.hex2color(mcolors.CSS4_COLORS['black']), mcolors.hex2color(mcolors.CSS4_COLORS['white'])], rng=1337, pastel_factor=0.5)
colors_icd_chpts = {icd_chpt: colors[icd_chpt_id] for icd_chpt_id, icd_chpt in enumerate(icd_chpts)}
colormaps_icd_chpts = {
    icd_chpt: LinearSegmentedColormap.from_list(
        name=f"ICD-11 Chapter {icd_chpt} cmap",
        colors=[make_rgb_transparent(colors_icd_chpts[icd_chpt], (1, 1, 1), 0.2), colors_icd_chpts[icd_chpt]], N=256
    )
    for icd_chpt in icd_chpts
}

# Prepare data for the figure
# SHAP values can be different due to randomization. Use previously calculated files
df_imm_shap = pd.read_excel(f"{path_data}/shap_mean_diff.xlsx", index_col=0)
df_imm_shap_table = df_imm_shap[[f'Chapter {icd_chpt}' for icd_chpt in icd_chpts]]
df_imm_shap_table.index = df_imm_shap_table.index.str.replace('_log', '')

# Plot Figure 6a
df_fig = df_imm_shap_table.astype(float)
x_ticks_colors = {f'Chapter {icd_chpt}': colors_icd_chpts[icd_chpt] for icd_chpt in icd_chpts}
sns.set_theme(style='ticks')
clustermap = sns.clustermap(
    df_fig,
    annot=True,
    col_cluster=False,
    row_cluster=True,
    fmt=".2f",
    center=0.0,
    cmap='seismic',
    linewidth=0.1,
    linecolor='black',
    tree_kws=dict(linewidths=1.5),
    figsize=(16, 12),
    cbar_kws={'orientation': 'horizontal'}
)
clustermap.ax_heatmap.set_xlabel('')
clustermap.ax_heatmap.set_ylabel('')
for spine in clustermap.ax_cbar.spines.values():
    spine.set(visible=True, lw=0.25, edgecolor="black")
clustermap.ax_heatmap.set_xticklabels(clustermap.ax_heatmap.get_xmajorticklabels(), rotation=0, path_effects=[pe.withStroke(linewidth=0.5, foreground="black")])
for tick_label in clustermap.ax_heatmap.get_xticklabels():
    tick_label.set_color(x_ticks_colors[tick_label.get_text()])
clustermap_pos = clustermap.ax_heatmap.get_position()
clustermap.ax_cbar.set_position([clustermap_pos.x0, clustermap_pos.y1 + 0.05, clustermap_pos.width, 0.03])
clustermap.ax_cbar.set_title("XAI age acceleration difference", fontsize='large')
clustermap.ax_cbar.tick_params(labelsize='large')
for spine in clustermap.ax_cbar.spines:
    clustermap.ax_cbar.spines[spine].set_linewidth(1)
plt.savefig(f"{path_plots}/figure6a.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_plots}/figure6a.pdf", bbox_inches='tight')
plt.close(clustermap.figure)